In [2]:
# We are gonna solve the SPAM/HAM Classification with Word2Vec and AvgWord2Vec

In [50]:
!pip install gensim

In [3]:
import gensim
from gensim.models import Word2Vec,KeyedVectors

In [4]:
import gensim.downloader as api
# wv=api.load("word2vec-google-news-300")
# Already loaded

In [5]:
# Storing it so I dont have to download the api again (api size 1GB)
import pickle
# with open("word2vec.pkl","wb") as fileobject:
#     pickle.dump(wv,fileobject)

In [6]:
# load from pickle file
with open("word2vec.pkl", "rb") as fileobject:
    wv=pickle.load(fileobject)

In [7]:
vec_king=wv["king"]

In [8]:
vec_king

array([ 1.25976562e-01,  2.97851562e-02,  8.60595703e-03,  1.39648438e-01,
       -2.56347656e-02, -3.61328125e-02,  1.11816406e-01, -1.98242188e-01,
        5.12695312e-02,  3.63281250e-01, -2.42187500e-01, -3.02734375e-01,
       -1.77734375e-01, -2.49023438e-02, -1.67968750e-01, -1.69921875e-01,
        3.46679688e-02,  5.21850586e-03,  4.63867188e-02,  1.28906250e-01,
        1.36718750e-01,  1.12792969e-01,  5.95703125e-02,  1.36718750e-01,
        1.01074219e-01, -1.76757812e-01, -2.51953125e-01,  5.98144531e-02,
        3.41796875e-01, -3.11279297e-02,  1.04492188e-01,  6.17675781e-02,
        1.24511719e-01,  4.00390625e-01, -3.22265625e-01,  8.39843750e-02,
        3.90625000e-02,  5.85937500e-03,  7.03125000e-02,  1.72851562e-01,
        1.38671875e-01, -2.31445312e-01,  2.83203125e-01,  1.42578125e-01,
        3.41796875e-01, -2.39257812e-02, -1.09863281e-01,  3.32031250e-02,
       -5.46875000e-02,  1.53198242e-02, -1.62109375e-01,  1.58203125e-01,
       -2.59765625e-01,  

In [9]:
vec_king.shape

(300,)

In [10]:
import pandas as pd
messages=pd.read_csv("../Dataset/SMSSpamCollection.txt",sep="\t",names=["label","message"])

In [11]:
from nltk.stem import WordNetLemmatizer
import re
lm=WordNetLemmatizer()

In [12]:
import nltk
from nltk.corpus import stopwords

In [13]:
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\agarw\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [14]:
corpus=[]
for i in range(len(messages)):
    sentence=re.sub("[^a-zA-Z]"," ",messages["message"][i])
    sentence=sentence.lower()
    words=sentence.split()

    words=[lm.lemmatize(word) for word in words] # Not removing stopwords for now / need to check vector creation for each word
    sentence=" ".join(words)
    corpus.append(sentence)

In [15]:
# corpus

In [16]:
len(corpus)

5572

In [17]:
from nltk.tokenize import sent_tokenize

In [18]:
for i, sent in enumerate(corpus):
    if len(sent_tokenize(sent)) == 0:
        print(i, repr(sent))

1612 ''
3376 ''
4824 ''


In [20]:
print(messages.iloc[[1612,3376,4824],:])

     label  message
1612   ham      645
3376   ham      :) 
4824   ham  :-) :-)


In [21]:
messages = messages.drop(index=[1612,3376,4824])

In [22]:
# Other way to do this
# Special Function simple_preprocess

from gensim.utils import simple_preprocess
words=[]
for sent in corpus:
    sent_token=sent_tokenize(sent)
    for sent in sent_token:
        words.append(simple_preprocess(sent))

In [23]:
len(words)

5569

In [24]:
from gensim.models import Word2Vec

In [25]:
## Lets train word2vec from scratch
model=Word2Vec(words)

In [26]:
# to get all the vocabulary
# model.wv.index_to_key

In [27]:
model.corpus_count

5569

In [28]:
model.epochs

5

In [29]:
model.wv.similar_by_word("good")

[('all', 0.9987724423408508),
 ('night', 0.9987261891365051),
 ('morning', 0.9986867904663086),
 ('well', 0.9986647963523865),
 ('nice', 0.9984946250915527),
 ('oh', 0.9984921216964722),
 ('happy', 0.9984858632087708),
 ('not', 0.9984335899353027),
 ('hope', 0.9984229803085327),
 ('thing', 0.9984197020530701)]

In [30]:
model.wv["good"].shape

(100,)

In [49]:
import numpy as np
def avg_word2vec(doc):
    vectors = [model.wv[word] for word in doc if word in model.wv.index_to_key]
    if len(vectors) > 0:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(model.vector_size)

In [50]:
!pip install tqdm

In [51]:
from tqdm import tqdm

In [52]:
X=[]
for i in tqdm(range(len(words))):
    X.append(avg_word2vec(words[i]))

100%|██████████| 5569/5569 [00:00<00:00, 10566.51it/s]


In [53]:
# X

In [54]:
len(X)

5569

In [55]:
# Independent Feature
X_new=np.array(X)

In [56]:
X_new.shape

(5569, 100)

In [57]:
X_new[0].shape

(100,)

In [58]:
## Dependent Feature
y=pd.get_dummies(messages["label"])
y=y.iloc[:,0].values

In [59]:
y.shape

(5569,)

In [60]:
## Independent Feature -- X_new
## Dependent Feature -- y

In [61]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X_new,y,test_size=0.20)

In [64]:
X_train.shape,X_test.shape,y_train.shape,y_test.shape

((4455, 100), (1114, 100), (4455,), (1114,))

In [65]:
## We will apply sklearn random forest classification
from sklearn.ensemble import RandomForestClassifier
classifier=RandomForestClassifier()

In [66]:
classifier.fit(X_train,y_train)

RandomForestClassifier()

In [67]:
y_pred=classifier.predict(X_test)

In [72]:
from sklearn.metrics import accuracy_score,classification_report
print("Accuracy Score = %.2f"%accuracy_score(y_test,y_pred))
print("Classification_report :")
print(classification_report(y_test,y_pred))

Accuracy Score = 0.97
Classification_report :
              precision    recall  f1-score   support

       False       0.90      0.89      0.89       148
        True       0.98      0.98      0.98       966

    accuracy                           0.97      1114
   macro avg       0.94      0.93      0.94      1114
weighted avg       0.97      0.97      0.97      1114

